# IMDb Review Preprocessing

This notebook prepares IMDb audience reviews for experience-based text analysis.

Reviews are restricted to titles with trusted IMDb-to-TMDB matches, connected to their TMDB entities, and minimally cleaned for downstream experience scoring.

The raw IMDb review files remain unchanged. The output of this notebook becomes the processed review layer used to derive experience-based descriptors.

## Workflow

1. Load the raw IMDb review dataset.
2. Load the trusted IMDb-to-TMDB crosswalk.
3. Keep reviews belonging to trusted titles.
4. Attach TMDB identifiers to each review.
5. Prepare review text for experience scoring.
6. Validate and save the processed review dataset.

## 0. Setup

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds


# Get the main project folder from the notebooks folder
project_root = Path.cwd().parent

# Folder containing the six raw IMDb review Parquet files
imdb_folder = (
    project_root
    / "data"
    / "raw"
    / "imdb_reviews"
)

# Trusted IMDb-to-TMDB matches created in notebook 03
crosswalk_file = (
    project_root
    / "data"
    / "processed"
    / "high_confidence_tmdb_matches.parquet"
)

# Treat the six IMDb Parquet files as one dataset
imdb_dataset = ds.dataset(
    imdb_folder,
    format="parquet"
)

print("IMDb dataset ready.")
print("Crosswalk:", crosswalk_file)

IMDb dataset ready.
Crosswalk: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\high_confidence_tmdb_matches.parquet


## 1. Load Trusted Title Crosswalk

Only IMDb titles with high-confidence TMDB matches are included in the downstream experience dataset.

In [2]:
# Load the trusted title mappings created during TMDB enrichment
trusted_matches = pd.read_parquet(crosswalk_file)

print(f"Trusted IMDb title mappings: {len(trusted_matches):,}")

trusted_matches[
    [
        "imdb_title",
        "tmdb_id",
        "media_type",
        "tmdb_title"
    ]
].head()

Trusted IMDb title mappings: 7,726


,imdb_title,tmdb_id,media_type,tmdb_title
0,Avengers: Endgame (2019),299534.0,movie,Avengers: Endgame
1,The Shawshank Redemption (1994),278.0,movie,The Shawshank Redemption
2,Dil Bechara (2020),645484.0,movie,Dil Bechara
3,Captain Marvel (2019),299537.0,movie,Captain Marvel
4,The Dark Knight (2008),155.0,movie,The Dark Knight


## 2. Filter Reviews to Trusted Titles

The full IMDb corpus contains reviews for more than 453,000 title strings. Only reviews belonging to the trusted IMDb-to-TMDB matches are needed for the downstream experience dataset.

PyArrow filters the Parquet dataset before the matching rows are loaded into memory.

In [3]:
# Get the IMDb title strings that have trusted TMDB matches
trusted_titles = trusted_matches["imdb_title"].dropna().unique().tolist()

# Define which review fields are needed downstream
review_columns = [
    "review_id",
    "movie",
    "rating",
    "review_summary",
    "review_date",
    "spoiler_tag",
    "review_detail",
    "helpful"
]

# Only scan reviews whose IMDb title appears in the trusted crosswalk
trusted_review_filter = ds.field("movie").isin(trusted_titles)

print(f"Trusted titles used for filtering: {len(trusted_titles):,}")

Trusted titles used for filtering: 7,726


### Validate the Title Filter

Before processing the full corpus, read a small batch of matching reviews to confirm that the IMDb title strings correctly connect to the trusted crosswalk.

In [4]:
# Create a scanner that reads only the needed columns and trusted titles
review_scanner = imdb_dataset.scanner(
    columns=review_columns,
    filter=trusted_review_filter,
    batch_size=50_000
)

# Read only the first matching batch so we can verify the result
first_batch = next(review_scanner.to_batches())
sample_reviews = first_batch.to_pandas()

print(f"Rows in test batch: {len(sample_reviews):,}")

sample_reviews.head()

Rows in test batch: 23,836


,review_id,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful
0,rw5704482,After Life (2019– ),9,Very Strong Season 2,3 May 2020,0,"I enjoyed the first season, but I must say I t...","[1, 1]"
1,rw5704483,The Valhalla Murders (2019– ),6,Icelandic detectives?,3 May 2020,0,I know Iceland is a small country and police d...,"[2, 2]"
2,rw5704484,Special OPS (2020– ),7,Nothing special,3 May 2020,0,"Except K K , no other actor looks comfortable ...","[0, 0]"
3,rw5704485,#BlackAF (2020– ),8,Good but,3 May 2020,0,I'm guessing that as a 62 year old white woman...,"[5, 9]"
4,rw5704488,All About Eve (1950),10,Amazing,3 May 2020,0,Having seen this film for the first time today...,"[0, 1]"


In [5]:
# Keep only the crosswalk fields needed to identify the final TMDB entity
title_crosswalk = trusted_matches[
    [
        "imdb_title",
        "tmdb_id",
        "media_type",
        "tmdb_title"
    ]
].copy()

# Connect the IMDb review title to its trusted TMDB entity
sample_reviews = sample_reviews.merge(
    title_crosswalk,
    left_on="movie",
    right_on="imdb_title",
    how="left"
)

print(
    "Reviews missing a TMDB ID:",
    sample_reviews["tmdb_id"].isna().sum()
)

sample_reviews[
    [
        "movie",
        "tmdb_id",
        "media_type",
        "tmdb_title",
        "review_summary"
    ]
].head()

Reviews missing a TMDB ID: 0


,movie,tmdb_id,media_type,tmdb_title,review_summary
0,After Life (2019– ),79410.0,tv,After Life,Very Strong Season 2
1,The Valhalla Murders (2019– ),97405.0,tv,The Valhalla Murders,Icelandic detectives?
2,Special OPS (2020– ),100612.0,tv,Special Ops,Nothing special
3,#BlackAF (2020– ),101200.0,tv,#blackAF,Good but
4,All About Eve (1950),705.0,movie,All About Eve,Amazing


## 3. Build the Curated Review Dataset

For each matching review, the IMDb title is connected to its TMDB entity. The review summary and full review are then combined into a single text field for downstream experience scoring.


In [6]:
# Count the reviews that belong to trusted IMDb-to-TMDB matches
trusted_review_count = imdb_dataset.count_rows(
    filter=trusted_review_filter
)

print(f"Trusted reviews to process: {trusted_review_count:,}")

Trusted reviews to process: 2,732,406


### Prepare the Processed Output

The filtered reviews are written in batches rather than loaded into memory all at once, keeping memory usage manageable while processing millions of review records.

In [7]:
# Folder for the processed review dataset
curated_reviews_dir = (
    project_root
    / "data"
    / "processed"
    / "curated_reviews"
)

curated_reviews_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Output folder:", curated_reviews_dir)

Output folder: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\curated_reviews


In [8]:
import re

# Replace repeated whitespace while keeping the original review language intact
def clean_review_text(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    return text


# Scan only the trusted reviews and fields needed downstream
review_scanner = imdb_dataset.scanner(
    columns=review_columns,
    filter=trusted_review_filter,
    batch_size=50_000
)

total_written = 0
file_number = 1

for batch in review_scanner.to_batches():

    reviews = batch.to_pandas()

    # Attach the trusted TMDB entity to each IMDb review
    reviews = reviews.merge(
        title_crosswalk,
        left_on="movie",
        right_on="imdb_title",
        how="left"
    )

    # Clean the two text fields without removing useful language
    reviews["review_summary"] = reviews["review_summary"].apply(
        clean_review_text
    )

    reviews["review_detail"] = reviews["review_detail"].apply(
        clean_review_text
    )

    # Combine the short summary and full review into one analysis field
    reviews["review_text"] = (
        reviews["review_summary"]
        + ". "
        + reviews["review_detail"]
    ).str.strip(". ")

    # The crosswalk copy of the IMDb title is no longer needed
    reviews = reviews.drop(columns=["imdb_title"])

    output_file = (
        curated_reviews_dir
        / f"part-{file_number:04d}.parquet"
    )

    # Save each processed batch as its own Parquet file
    reviews.to_parquet(
        output_file,
        index=False
    )

    total_written += len(reviews)

    if file_number % 10 == 0:
        print(f"{total_written:,} reviews written")

    file_number += 1

print()
print(f"TOTAL REVIEWS WRITTEN: {total_written:,}")
print(f"PARQUET FILES CREATED: {file_number - 1:,}")

240,154 reviews written
492,490 reviews written
732,882 reviews written
983,036 reviews written
1,213,726 reviews written
1,454,767 reviews written
1,687,727 reviews written
1,951,192 reviews written
2,168,181 reviews written
2,362,880 reviews written
2,577,062 reviews written

TOTAL REVIEWS WRITTEN: 2,732,406
PARQUET FILES CREATED: 115


## Validate the processed reviews

Do a few checks to make sure the saved review dataset is complete and ready for experience scoring.

In [9]:
# Treat all of the processed parquet files as one dataset
curated_dataset = ds.dataset(
    curated_reviews_dir,
    format="parquet"
)

# Check that the number of saved rows matches what we processed
saved_review_count = curated_dataset.count_rows()

print(f"Expected reviews: {trusted_review_count:,}")
print(f"Saved reviews:    {saved_review_count:,}")


# Load only the fields needed for the remaining checks
validation_df = curated_dataset.to_table(
    columns=["review_id", "tmdb_id", "review_text"]
).to_pandas()

# Check for problems that would affect experience scoring
missing_tmdb = validation_df["tmdb_id"].isna().sum()
duplicate_reviews = validation_df["review_id"].duplicated().sum()
blank_review_text = (
    validation_df["review_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"\nMissing TMDB IDs:     {missing_tmdb:,}")
print(f"Duplicate review IDs: {duplicate_reviews:,}")
print(f"Blank review text:    {blank_review_text:,}")

Expected reviews: 2,732,406
Saved reviews:    2,732,406

Missing TMDB IDs:     0
Duplicate review IDs: 9,170
Blank review text:    9


### Check duplicate reviews

Some review IDs appear more than once in the processed data. Check whether those duplicates have the same title and review text before removing them.

In [11]:
# Pull all rows involved in a duplicated review ID
duplicate_rows = validation_df[
    validation_df["review_id"].duplicated(keep=False)
].copy()

# Check whether the same review ID ever has different TMDB IDs or review text
duplicate_check = (
    duplicate_rows
    .groupby("review_id")
    .agg(
        row_count=("review_id", "size"),
        tmdb_id_count=("tmdb_id", "nunique"),
        review_text_count=("review_text", "nunique")
    )
)

conflicting_duplicates = duplicate_check[
    (duplicate_check["tmdb_id_count"] > 1)
    | (duplicate_check["review_text_count"] > 1)
]

print(f"Unique duplicated review IDs: {len(duplicate_check):,}")
print(f"Rows involved in duplicates:  {len(duplicate_rows):,}")
print(f"Conflicting duplicate IDs:    {len(conflicting_duplicates):,}")

Unique duplicated review IDs: 9,170
Rows involved in duplicates:  18,340
Conflicting duplicate IDs:    6


In [12]:
# Show the duplicate IDs where the title or review text does not match
conflicting_ids = conflicting_duplicates.index

conflicting_rows = duplicate_rows[
    duplicate_rows["review_id"].isin(conflicting_ids)
].sort_values("review_id")

conflicting_rows

,review_id,tmdb_id,review_text
1639383,rw1091803,15237.0,Musical Format = Artistic Freedoms. I'm a fini...
2459259,rw1091803,15237.0,Musical Format = Artistic Freedoms. I'm a fini...
1698072,rw2044573,96.0,"""The Heat's Back On"".. But Not Hot !. A kind o..."
2226897,rw2044573,96.0,"""The Heat's Back On"".. But Not Hot !. A kind o..."
1693740,rw3045667,34415.0,Soap opera Seattle. I'm a Seattle resident so ...
2329335,rw3045667,34415.0,Soap opera Seattle. I'm a Seattle resident so ...
423354,rw6443554,113627.0,She deserves better but not bad. I just got th...
484683,rw6443554,113627.0,She deserves better but not bad gets funnier. ...
427831,rw6446869,138843.0,"Not bad, but very overrated. This movie is jus..."
486124,rw6446869,138843.0,"Not bad, but very overrated. This movie is jus..."


### Clean duplicate and empty reviews

Most duplicate review IDs are exact copies, so only one copy is kept. A few IDs have different review text between copies, so those are removed instead of choosing one version. Reviews with no usable text are also removed.

In [13]:
# IDs where both copies disagree
conflicting_id_set = set(conflicting_ids)

# The rest of the duplicate IDs are safe to deduplicate
normal_duplicate_ids = (
    set(duplicate_check.index)
    - conflicting_id_set
)

print(f"Duplicate IDs to deduplicate: {len(normal_duplicate_ids):,}")
print(f"Conflicting IDs to remove:    {len(conflicting_id_set):,}")

Duplicate IDs to deduplicate: 9,164
Conflicting IDs to remove:    6


In [14]:
# Folder for the final review dataset used in scoring
clean_reviews_dir = (
    project_root
    / "data"
    / "processed"
    / "clean_reviews"
)

clean_reviews_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Track only duplicate IDs we have already kept
seen_duplicate_ids = set()

total_read = 0
total_written = 0

# Process the existing parquet files in order
curated_files = sorted(curated_reviews_dir.glob("part-*.parquet"))

for file_number, file_path in enumerate(curated_files, start=1):

    reviews = pd.read_parquet(file_path)
    total_read += len(reviews)

    # Remove the few IDs where the duplicate copies disagree
    reviews = reviews[
        ~reviews["review_id"].isin(conflicting_id_set)
    ].copy()

    # Remove reviews that have no usable text
    reviews = reviews[
        reviews["review_text"]
        .fillna("")
        .str.strip()
        .ne("")
    ].copy()

    # Only the known duplicate IDs need special handling
    duplicate_mask = reviews["review_id"].isin(normal_duplicate_ids)

    regular_rows = reviews[~duplicate_mask]
    duplicate_rows_current = reviews[duplicate_mask].copy()

    # Keep a duplicated review only the first time we see its ID
    keep_duplicate = ~duplicate_rows_current["review_id"].isin(
        seen_duplicate_ids
    )

    duplicate_rows_current = duplicate_rows_current[
        keep_duplicate
    ]

    seen_duplicate_ids.update(
        duplicate_rows_current["review_id"]
    )

    # Put the unique rows and kept duplicate rows back together
    cleaned_reviews = pd.concat(
        [regular_rows, duplicate_rows_current],
        ignore_index=True
    )

    output_file = (
        clean_reviews_dir
        / f"part-{file_number:04d}.parquet"
    )

    cleaned_reviews.to_parquet(
        output_file,
        index=False
    )

    total_written += len(cleaned_reviews)

    if file_number % 20 == 0:
        print(f"{file_number} files processed")


print()
print(f"TOTAL ROWS READ:    {total_read:,}")
print(f"TOTAL ROWS WRITTEN: {total_written:,}")
print(f"ROWS REMOVED:       {total_read - total_written:,}")

20 files processed
40 files processed
60 files processed
80 files processed
100 files processed

TOTAL ROWS READ:    2,732,406
TOTAL ROWS WRITTEN: 2,723,235
ROWS REMOVED:       9,171


## Final checks

Check the cleaned dataset one more time before using it for experience scoring.

In [15]:
# Treat the cleaned parquet files as one dataset
clean_dataset = ds.dataset(
    clean_reviews_dir,
    format="parquet"
)

clean_review_count = clean_dataset.count_rows()

# Only load the fields needed for validation
clean_validation = clean_dataset.to_table(
    columns=["review_id", "tmdb_id", "review_text"]
).to_pandas()

remaining_duplicates = clean_validation["review_id"].duplicated().sum()
missing_tmdb = clean_validation["tmdb_id"].isna().sum()
blank_text = (
    clean_validation["review_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Clean review count:       {clean_review_count:,}")
print(f"Remaining duplicate IDs: {remaining_duplicates:,}")
print(f"Missing TMDB IDs:        {missing_tmdb:,}")
print(f"Blank review text:       {blank_text:,}")

Clean review count:       2,723,235
Remaining duplicate IDs: 14
Missing TMDB IDs:        0
Blank review text:       0


### Remove the remaining duplicates

A few duplicate IDs were in the same Parquet file and survived the first cleanup pass. Remove the extra copies so each review ID appears only once.

In [16]:
# Get the review IDs that are still duplicated
remaining_duplicate_ids = set(
    clean_validation.loc[
        clean_validation["review_id"].duplicated(keep=False),
        "review_id"
    ]
)

print(f"Duplicate IDs left to fix: {len(remaining_duplicate_ids):,}")

Duplicate IDs left to fix: 14


In [17]:
# Track which remaining duplicate IDs have already been kept
seen_ids = set()

clean_files = sorted(clean_reviews_dir.glob("part-*.parquet"))

rows_removed = 0

for file_path in clean_files:

    reviews = pd.read_parquet(file_path)

    # Only these few IDs still need duplicate handling
    duplicate_mask = reviews["review_id"].isin(remaining_duplicate_ids)

    regular_rows = reviews[~duplicate_mask]
    duplicate_rows_current = reviews[duplicate_mask].copy()

    # If both copies are in this file, keep only one
    duplicate_rows_current = duplicate_rows_current.drop_duplicates(
        subset="review_id",
        keep="first"
    )

    # If the ID was already kept in an earlier file, remove it here
    duplicate_rows_current = duplicate_rows_current[
        ~duplicate_rows_current["review_id"].isin(seen_ids)
    ]

    seen_ids.update(duplicate_rows_current["review_id"])

    cleaned_file = pd.concat(
        [regular_rows, duplicate_rows_current],
        ignore_index=True
    )

    rows_removed += len(reviews) - len(cleaned_file)

    # Replace the file with the corrected version
    cleaned_file.to_parquet(
        file_path,
        index=False
    )

print(f"Additional duplicate rows removed: {rows_removed:,}")

Additional duplicate rows removed: 14



Make sure the cleaned review dataset has the expected row count and no remaining issues before moving on to experience scoring.

In [18]:
# Reload the cleaned parquet files after the duplicate fix
clean_dataset = ds.dataset(
    clean_reviews_dir,
    format="parquet"
)

final_review_count = clean_dataset.count_rows()

# Only load the columns needed for the final checks
final_validation = clean_dataset.to_table(
    columns=["review_id", "tmdb_id", "review_text"]
).to_pandas()

remaining_duplicates = final_validation["review_id"].duplicated().sum()
missing_tmdb = final_validation["tmdb_id"].isna().sum()
blank_text = (
    final_validation["review_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Final review count:       {final_review_count:,}")
print(f"Remaining duplicate IDs: {remaining_duplicates:,}")
print(f"Missing TMDB IDs:        {missing_tmdb:,}")
print(f"Blank review text:       {blank_text:,}")

Final review count:       2,723,221
Remaining duplicate IDs: 0
Missing TMDB IDs:        0
Blank review text:       0


## Result

The final review dataset contains 2,723,221 cleaned reviews linked to trusted TMDB titles. Duplicate review IDs, conflicting duplicates, and empty review text were removed.

These reviews are ready to be used for experience scoring.